# Interpolations

L’interpolation est une technique mathématique qui permet d’estimer des valeurs manquantes ou intermédiaires à partir d’un ensemble de points connus. L’idée est de construire une fonction qui relie ces points, de façon à pouvoir calculer des valeurs pour n’importe quel point situé entre eux.

## Imports & Variables

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
valeur_de_travail = 'T_Q'

fichier_nappe = "../../../data/fusion/data_03288X0042_P.csv"
dossier_nappe = "../../../data/fusion"

df = pd.read_csv(fichier_nappe, sep=";")
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time')


df[valeur_de_travail] = pd.to_numeric(df[valeur_de_travail], errors='coerce')

## Interpolation linéaire

L’interpolation linéaire permet d’estimer des valeurs manquantes en supposant que l’évolution entre deux points connus suit une droite. Les valeurs manquantes sont donc calculées à partir de la droite reliant les observations voisine

$$
f(x) = ax + b
$$

où :

- $a$ représente la pente 
- $b$ l’ordonnée à l’origine

In [ ]:
def interpolation_lineaire_array(df:pd.DataFrame, valeur_de_travail:str)->np.ndarray:
    """Interpole linéairement la colonne `valeur_de_travail`

    Args:
        df (pd.DataFrame): Dataset à compléter
        valeur_de_travail (str): Valeur que l'on veut interpolé

    Returns:
        np.ndarray: Le dataset complété sur la valeur demandée
    """

    return df[valeur_de_travail].interpolate(method='linear', limit_direction='both').to_numpy()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['time'], interpolation_lineaire_array(df, valeur_de_travail), label="Prédiction")
plt.plot(df['time'], df[valeur_de_travail], label="Mesure")
plt.xlabel("Date")
plt.ylabel(valeur_de_travail)
plt.legend()
plt.show()

## Interpolation via spline cubique

L’interpolation cubique estime les valeurs manquantes en utilisant un polynôme de degré 3 entre les points connus. Contrairement à l’interpolation linéaire, cette méthode permet d’obtenir une courbe plus lisse et de mieux capturer les variations non linéaires des données.

$$
f(x) = ax^3 + bx^2 + cx + d
$$

où :

- $a$, $b$, $c$ et $d$ sont des coefficients déterminés à partir des points connus afin d'ajuster le polynôme aux données.

In [ ]:
def interpolation_cubique_array(df:pd.DataFrame, valeur_de_travail:str)->np.ndarray:
    """Interpole cubiquement la colonne `valeur_de_travail`

    Args:
        df (pd.DataFrame): Dataset à compléter
        valeur_de_travail (str): Valeur que l'on veut interpolé

    Returns:
        np.ndarray: Le dataset complété sur la valeur demandée
    """

    return df[valeur_de_travail].interpolate(method='cubic').to_numpy()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['time'], interpolation_cubique_array(df, valeur_de_travail), label="Prédiction")
plt.plot(df['time'], df[valeur_de_travail], label="Mesure")
plt.xlabel("Date")
plt.ylabel(valeur_de_travail)
plt.legend()
plt.show()

## Interpolation via Polynômes de degré n

L’interpolation polynomiale estime les valeurs manquantes en utilisant un polynôme de degré $n$ ajusté aux points connus.

$$
f(x) = a_n x^n + a_{n-1} x^{n-1} + \dots + a_1 x + a_0
$$

où :

- $n$ est le degré du polynôme,
- $a_0, a_1, \dots, a_n$ sont des coefficients déterminés à partir des points connus.

In [ ]:
def interpolation_polynomiale_array(df:pd.DataFrame, valeur_de_travail:str, deg:int=2)->np.ndarray:
    """Interpole polynomialement la colonne `valeur_de_travail`

    Args:
        df (pd.DataFrame): Dataset à compléter
        valeur_de_travail (str): Valeur que l'on veut interpolé
        deg (int): Degré de la polynomiale

    Returns:
        np.ndarray: Le dataset complété sur la valeur demandée
    """

    return df[valeur_de_travail].interpolate(method="polynomial", order=deg).to_numpy()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['time'], interpolation_polynomiale_array(df, valeur_de_travail), label="Prédiction")
plt.plot(df['time'], df[valeur_de_travail], label="Mesure")
plt.xlabel("Date")
plt.ylabel(valeur_de_travail)
plt.legend()
plt.show()

## Interpolation via B-spline

L’interpolation par spline estime les valeurs manquantes en utilisant des polynômes de degré $n$ définis sur plusieurs intervalles entre les points connus. Cela permet d’obtenir une courbe plus lisse et plus stable qu’une interpolation polynomiale globale.

$$
f(x) = ax^3 + bx^2 + cx + d
$$

où :

- $a$, $b$, $c$ et $d$ sont des coefficients déterminés à partir des points connus.

In [ ]:
def interpolation_spline_array(df:pd.DataFrame, valeur_de_travail:str, deg:int=3)->np.ndarray:
    """Interpole via une spline polynomiale par moindres carrés la colonne `valeur_de_travail`

    Args:
        df (pd.DataFrame): Dataset à compléter
        valeur_de_travail (str): Valeur que l'on veut interpolé
        deg (int): Degré de la spline

    Returns:
        np.ndarray: Le dataset complété sur la valeur demandée
    """

    return df[valeur_de_travail].interpolate(method="spline", order=deg).to_numpy()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['time'], interpolation_spline_array(df, valeur_de_travail), label="Prédiction")
plt.plot(df['time'], df[valeur_de_travail], label="Mesure")
plt.xlabel("Date")
plt.ylabel(valeur_de_travail)
plt.legend()
plt.show()

## Interpolation via PCHIP

L’interpolation PCHIP (Piecewise Cubic Hermite Interpolating Polynomial) utilise des polynômes cubiques **par morceaux** pour estimer les valeurs manquantes, en préservant la **monotonie** et les **formes locales** de la série.  

$$
f(x) = S_i(x), \quad x \in [x_i, x_{i+1}]
$$

où :

- $S_i(x)$ est un polynôme cubique défini sur chaque intervalle $[x_i, x_{i+1}]$.

In [ ]:
def interpolation_pchip_array(df:pd.DataFrame, valeur_de_travail:str)->np.ndarray:
    """Interpole via PCHIP la colonne `valeur_de_travail`

    Args:
        df (pd.DataFrame): Dataset à compléter
        valeur_de_travail (str): Valeur que l'on veut interpolé

    Returns:
        np.ndarray: Le dataset complété sur la valeur demandée
    """

    return df[valeur_de_travail].interpolate(method='pchip').to_numpy()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['time'], interpolation_pchip_array(df, valeur_de_travail), label="Prédiction")
plt.plot(df['time'], df[valeur_de_travail], label="Mesure")
plt.xlabel("Date")
plt.ylabel(valeur_de_travail)
plt.legend()
plt.show()

## Interpolation via akima

L’interpolation Akima utilise également des polynômes cubiques **par morceaux**, mais avec un calcul spécial des pentes pour éviter les oscillations excessives. Elle est utile pour les données présentant des **variations abruptes**.  

$$
f(x) = A_i(x), \quad x \in [x_i, x_{i+1}]
$$

où :

- $A_i(x)$ est le polynôme cubique sur l’intervalle $[x_i, x_{i+1}]$ avec des pentes calculées selon la méthode d’Akima¹.

méthode d'Akima¹ : Les pentes aux points intermédiaires ne sont pas simplement calculées par la pente linéaire des segments voisins, mais selon un moyen pondéré des pentes locales des segments adjacents 

In [ ]:
def interpolation_akima_array(df:pd.DataFrame, valeur_de_travail:str)->np.ndarray:
    """Interpole via Akima la colonne `valeur_de_travail`

    Args:
        df (pd.DataFrame): Dataset à compléter
        valeur_de_travail (str): Valeur que l'on veut interpolé

    Returns:
        np.ndarray: Le dataset complété sur la valeur demandée
    """

    return df[valeur_de_travail].interpolate(method='akima').to_numpy()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['time'], interpolation_akima_array(df, valeur_de_travail), label="Prédiction")
plt.plot(df['time'], df[valeur_de_travail], label="Mesure")
plt.xlabel("Date")
plt.ylabel(valeur_de_travail)
plt.legend()
plt.show()